# Сравнение результатов модели у предобработанных и непредобработанных данных

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

## Задача 1. Выделение обучающей и тестовой выборки

В нашем исследовании будут участвовать два датасета из прошлого урока:
- В первом датасете выбросы уже обработаны — к нему добавим масштабирование.
- Во втором выбросы (кроме пропусков целевой переменной) сохраним без изменений — и масштабирование применять не будем.

В нашем исследовании будут участвовать два датасета из прошлого урока:

- В первом датасете выбросы уже обработаны — к нему добавим масштабирование.
- Во втором выбросы (кроме пропусков целевой переменной) сохраним без изменений — и масштабирование применять не будем.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Загружаем данные
# В уроке есть датасет где мы удалили пропуски и удалили выбросы. И мы разбираем как помогает масштабирование

##df_transformed = pd.read_csv('turtles_t2_l5_no_outliers.csv')
##df_original = pd.read_csv('turtles_t2_l5_with_outliers.csv')

df_transformed = pd.read_csv("../data/turtles.csv", sep='\t', decimal=',')
df_original = pd.read_csv("../data/turtles.csv", sep='\t', decimal=',')

# Разделите признаки и целевую переменную
X_transformed = df_transformed.drop('weight', axis=1)
y_transformed = df_transformed['weight']
X_original = df_original.drop('weight', axis=1)
y_original = df_original['weight']

# Разделите df_transformed на train и test
# Используйте test_size=0.2 и random_state=627
X_train_transformed, X_test_transformed, y_train_transformed, y_test_transformed = train_test_split(
    X_transformed, y_transformed, test_size=0.2, random_state=627, shuffle=True  )

# Теперь выделите те же записи для df_original
X_train_original = X_original.loc[X_train_transformed.index]
X_test_original =  X_original.loc[X_test_transformed.index]
y_train_original = y_original.loc[y_train_transformed.index]
y_test_original = y_original.loc[y_test_transformed.index]

# Выведем размеры, чтобы убедиться, что всё совпадает
print("Размеры выборок (transformed):", X_train_transformed.shape, X_test_transformed.shape)
print("Размеры выборок (original):", X_train_original.shape, X_test_original.shape)

Размеры выборок (transformed): (7088, 19) (1773, 19)
Размеры выборок (original): (7088, 19) (1773, 19)


## Задача 2. Создание полных пайплайнов
Ч
тобы быстро обучить несколько моделей, создадим пайплайны, объединяющие этапы обработки и моделирования. Нам понадобятся две версии:
- пайплайн с полной обработкой данных;
- пайплайн с частичной обработкой (без масштабирования).

Задание 2

Создайте трансформер для полной обработки данных, аналогичный тому, что вы реализовали в прошлом уроке. Напомним, что он состоит из трёх частей:
- пайплайн для категориальных признаков (заполнение пропусков и кодирование категорий);
- пайплайн для числовых признаков (заполнение пропусков и масштабирование);
- заполнение пропусков для special_features.

Создайте трансформер для частичной обработки признаков. В отличие от полного трансформера, в нём не должно быть этапа с масштабированием числовых значений.

Создайте пайплайн для каждого варианта обработки, состоящий из двух шагов:
- Трансформер для обработки признаков (созданный в пункте 1 или 2). Назовите этот шаг preprocessor.
- Модель линейной регрессии с регуляризацией — Ridge(). Назовите этот шаг model.

In [ ]:
# Зададим группы признаков
cat_features = ['binomial_name']
num_features = [
    'shell_length', 'shell_width', 'head_length', 'head_width',
    'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',
    'circle_count', 'measure_count'
]
special_features = ['shell_crack']

## Создание пайплайнов
imputer = SimpleImputer(strategy="most_frequent")
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

num_sampler = SimpleImputer(strategy="median")
num_scaler = StandardScaler()

zero_imputer = SimpleImputer(strategy="constant", fill_value=0)

cat_pipeline = Pipeline(
    steps=[
        ('imputer', imputer),
        ('encoder', encoder)
    ]
)

num_pipeline_full = Pipeline(
    steps=[
        ('imputer', num_sampler),
        ('scaler', num_scaler),
    ]
)


# Допишите код трансформера с полной обработкой
full_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_features),
        ('num', num_pipeline_full, num_features),
        ('zero', zero_imputer, special_features),
    ]
)

# Допишите код трансформера с частичной обработкой
light_preprocessor =  ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_features),
        ('num', num_sampler, num_features),
        ('zero', zero_imputer, special_features),
    ]
)

# Создайте пайплайн для полной обработки
pipe_transformed = Pipeline(
    steps=[
        ('preprocessor', full_preprocessor),
        ('model', Ridge())
    ]
)

# Создайте пайплайн для частичной обработки
pipe_original = Pipeline(
    steps=[
        ('preprocessor', light_preprocessor),
        ('model', Ridge())
    ]
)


# Примените трансформеры к X_train_transformed и X_train_original

X_train_transformed_features = full_preprocessor.fit_transform(X_train_transformed)
feature_names_transformed = full_preprocessor.get_feature_names_out()
X_train_transformed_new = pd.DataFrame(X_train_transformed_features, columns=feature_names_transformed)

print("Данные без выбросов после трансформации:")
print(X_train_transformed_new.head())

X_train_original_features = light_preprocessor.fit_transform(X_train_original)
feature_names_original = light_preprocessor.get_feature_names_out()
X_train_original_new = pd.DataFrame(X_train_original_features, columns=feature_names_original)

print("\nДанные с выбросами после трансформации:")
print(X_train_original_new.head())

Данные без выбросов после трансформации:
   cat_pipeline__binomial_name_CARETTA CARETTA  \
0                                          0.0   
1                                          0.0   
2                                          1.0   
3                                          0.0   
4                                          0.0   

   cat_pipeline__binomial_name_CHELONIA MYDAS  \
0                                         0.0   
1                                         0.0   
2                                         0.0   
3                                         0.0   
4                                         0.0   

   cat_pipeline__binomial_name_Caretta Caretta  \
0                                          0.0   
1                                          0.0   
2                                          0.0   
3                                          0.0   
4                                          0.0   

   cat_pipeline__binomial_name_Caretta caretta  \
0           

## Задача 3. Обучение моделей и расчёт метрик


Задание 3

Напишите функцию train_eval_model, которая принимает пять параметров:
обучающие и тестовые данные ( X_train_transformed, y_train_transformed, X_test_transformed, y_test_transformed или X_train_original, y_train_original, X_test_original, y_test_original ).

пайплайн для обработки и моделирования ( pipe_transformed или pipe_original ).

Функция должна выполнять следующие шаги: 
- Обучить пайплайн на трейне.
- Получить предсказания на трейне и на тесте.
- Посчитать метрики MAE, RMSE и R².
- Вернуть обученный пайплайн и словарь с метриками.
- Обучите линейную регрессию на каждом датасете, вызвав train_eval_model с нужными данными и пайплайном. 

Сохраните посчитанные метрики в переменные metrics_transformed и metrics_original.
Сравните результаты.

In [ ]:
def train_eval_model(X_train, y_train, X_test, y_test, pipeline):

    # Допишите код функции
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    rmse = root_mean_squared_error(y_test, predictions)
 

    metrics = {"MAE": mae, "RMSE": rmse, "R2": r2}
    return pipeline, metrics


# Вызовите train_eval_model для двух вариантов данных и сохраните результаты 
pipeline_transformed_fitted, metrics_transformed = train_eval_model( X_train_transformed, y_train_transformed, X_test_transformed, y_test_transformed, pipe_transformed)
pipeline_original_fitted, metrics_original = train_eval_model(X_train_original, y_train_original, X_test_original, y_test_original, pipe_original )


# Выведем значения метрик
report = pd.DataFrame([metrics_transformed, metrics_original],
                      index=['Transformed (full preprocess)', 'Original (light preprocess)'])
print(report)

# Сравним коэффициенты моделей
coef_transformed = pd.Series(
    pipeline_transformed_fitted.named_steps['model'].coef_,
    index=feature_names_transformed
)
coef_original = pd.Series(
    pipeline_original_fitted.named_steps['model'].coef_,
    index=feature_names_original
)
print("Сумма коэффициентов по модулю для модели, обученной на полностью обработанных данных:", coef_transformed.apply(abs).sum())
print("Сумма коэффициентов по модулю для модели, обученной на частично обработанных данных:", coef_original.apply(abs).sum())

ValueError: Input y contains NaN.